In [1]:
import json
from dataclasses import dataclass, field
from typing import Optional
import os

from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field

In [2]:
load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# from app.services.question_generator import generate_question

In [3]:
class GeneratedQuestion(BaseModel):
    question: str
    topic: str
    difficulty: str
    question_type: str
    expected_concepts: list[str] = Field(
        default_factory=list
    )

class AnswerEvaluation(BaseModel):
    overall_score: float
    technical_accuracy: float
    depth: float
    reasoning: float
    clarity: float
    communication: float
    confidence: float
    strengths: list[str] = Field(default_factory=list)
    weaknesses: list[str] = Field(default_factory=list)
    should_challenge: bool = False
    suggested_follow_up: str = ""
    missing_concepts: list[str] = Field(default_factory=list)

In [4]:
with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate = json.load(f)

with open(
    "../data/interview_blueprint.json",
    "r",
    encoding="utf-8"
) as f:
    blueprint = json.load(f)

print("Target role:", blueprint["target_role"])

Target role: Data Scientist


In [5]:
@dataclass
class InterviewState:

    target_role: str

    questions_asked: list[str] = field(
        default_factory=list
    )

    answers: list[str] = field(
        default_factory=list
    )

    evaluations: list[dict] = field(
        default_factory=list
    )

    topics_covered: list[str] = field(
        default_factory=list
    )

    current_topic: Optional[str] = None

    current_difficulty: str = "medium"

    current_question: Optional[dict] = None

    time_remaining: int = 900

    max_questions: int = 12

    interview_status: str = "not_started"

    follow_up_count: int = 0

    question_count: int = 0

In [6]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

print(state)

InterviewState(target_role='Data Scientist', questions_asked=[], answers=[], evaluations=[], topics_covered=[], current_topic=None, current_difficulty='medium', current_question=None, time_remaining=900, max_questions=12, interview_status='not_started', follow_up_count=0, question_count=0)


In [7]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [8]:
DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard"
]
def adjust_difficulty(
    current_difficulty,
    answer_state
):

    current_index = DIFFICULTY_LEVELS.index(
        current_difficulty
    )

    if answer_state == "strong":

        new_index = min(
            current_index + 1,
            len(DIFFICULTY_LEVELS) - 1
        )

    elif answer_state in [
        "technical_gap",
        "shallow"
    ]:

        new_index = max(
            current_index - 1,
            0
        )

    else:

        new_index = current_index

    return DIFFICULTY_LEVELS[new_index]



In [9]:
def choose_next_topic(
    state,
    blueprint
):

    priority_topics = blueprint[
        "priority_topics"
    ]

    for topic in priority_topics:

        if topic not in state.topics_covered:
            return topic

    # fallback
    for topic in priority_topics:
        return topic

    return "General"

In [10]:
topic = choose_next_topic(
    state,
    blueprint
)

print(topic)

Statistics


In [11]:
def should_follow_up(evaluation):

    if evaluation["should_challenge"]:
        return True

    if evaluation["state"] in [
        "shallow",
        "weak_reasoning"
    ]:
        return True

    return False

In [12]:
MAX_FOLLOW_UPS = 2

def can_follow_up(state):

    return (
        state.follow_up_count
        < MAX_FOLLOW_UPS
    )

In [13]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [14]:
def manager_decision(
    state,
    evaluation=None,
    blueprint=None
):
    
    if should_end_interview(state):

        state.interview_status = "completed"

        return {
            "action": "finish",
            "reason": "Interview limit reached"
        }
    
    # Interview hasn't started
    if state.interview_status == "not_started":

        state.interview_status = "in_progress"

        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": "Start interview"
        }

    # We have an evaluation
    if evaluation is not None:

        answer_state = determine_answer_state(
            evaluation
        )

        # Challenge / follow-up
        if (
            should_follow_up(evaluation)
            and can_follow_up(state)
        ):

            state.follow_up_count += 1

            return {
                "action": "follow_up",
                "topic": state.current_topic,
                "difficulty": state.current_difficulty,
                "reason": answer_state
            }

        # Reset follow-up counter
        state.follow_up_count = 0

        # Adapt difficulty
        state.current_difficulty = (
            adjust_difficulty(
                state.current_difficulty,
                answer_state
            )
        )

        # Choose new topic
        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": answer_state
        }

    return {
        "action": "ask_question",
        "topic": choose_next_topic(
            state,
            blueprint
        ),
        "difficulty": state.current_difficulty,
        "reason": "Default"
    }

In [15]:
decision = manager_decision(
    state,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'Start interview'}


In [16]:
strong_evaluation = {
    "overall_score": 9,
    "technical_accuracy": 9,
    "depth": 9,
    "reasoning": 8,
    "should_challenge": False,
    "state": "strong"
}

decision = manager_decision(
    state,
    evaluation=strong_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'hard', 'reason': 'strong'}


In [17]:
weak_evaluation = {
    "overall_score": 4,
    "technical_accuracy": 4,
    "depth": 3,
    "reasoning": 4,
    "should_challenge": False,
    "state": "technical_gap"
}

decision = manager_decision(
    state,
    evaluation=weak_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'technical_gap'}


In [18]:
shallow_evaluation = {
    "overall_score": 6,
    "technical_accuracy": 7,
    "depth": 4,
    "reasoning": 4,
    "should_challenge": False,
    "state": "shallow"
}

decision = manager_decision(
    state,
    evaluation=shallow_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'follow_up', 'topic': None, 'difficulty': 'medium', 'reason': 'shallow'}


In [19]:
suspicious_evaluation = {
    "overall_score": 7,
    "technical_accuracy": 7,
    "depth": 6,
    "reasoning": 6,
    "should_challenge": True,
    "state": "acceptable"
}

decision = manager_decision(
    state,
    evaluation=suspicious_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'follow_up', 'topic': None, 'difficulty': 'medium', 'reason': 'acceptable'}


In [20]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [21]:
state.question_count = 12

print(
    should_end_interview(state)
)

True


In [22]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

decision = manager_decision(
    state,
    blueprint=blueprint
)
print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'Start interview'}


In [23]:
def generate_question(
    topic: str,
    difficulty: str,
    question_type: str,
    previous_questions: list[str] | None = None
):
    
    if previous_questions is None:
        previous_questions = []

    previous_text = "\n".join(
        f"- {q}"
        for q in previous_questions[-10:]
    )

    prompt = f"""
You are an expert technical interviewer.

Generate ONE interview question.

Target topic:
{topic}

Difficulty:
{difficulty}

Question type:
{question_type}

Previously asked questions:
{previous_text if previous_text else "None"}

Rules:

1. The question must test the specified topic.
2. Match the requested difficulty.
3. Do not repeat or closely rephrase previous questions.
4. The question should be appropriate for a technical interview.
5. Return expected concepts that a strong answer should contain.
6. Return ONLY valid JSON.

Return:

{{
    "question": "...",
    "topic": "...",
    "difficulty": "...",
    "question_type": "...",
    "expected_concepts": [
        "...",
        "..."
    ]
}}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert technical interviewer. "
                    "Return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return GeneratedQuestion.model_validate(result)

In [24]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

In [25]:
print(question.question)
print(question.expected_concepts)

You are running an A/B test to compare the conversion rates of two website designs. Design A received 2,000 visits with 120 conversions, and Design B received 2,500 visits with 165 conversions. Describe how you would test whether the difference in conversion rates is statistically significant, including the choice of test, assumptions, calculation steps, and interpretation of the results.
['Formulating null and alternative hypotheses for proportions', 'Choosing an appropriate test (e.g., two‑proportion z‑test or chi‑square test)', 'Checking assumptions such as independent samples and sufficient sample size for normal approximation', 'Calculating pooled proportion, standard error, and test statistic', 'Computing p‑value and comparing it to a significance level (e.g., α = 0.05)', 'Interpreting the result in terms of statistical significance and practical significance', 'Discussing Type I and Type II errors and test power']


In [26]:
candidate_answer = """
I chose XGBoost because it is a gradient boosting algorithm
that works very well with structured and tabular data. It can
capture nonlinear relationships and interactions between
features. It also provides useful feature importance and
usually performs well without requiring extensive feature
scaling.
"""

In [27]:
question_for_evaluation = question.model_dump()
print(
    json.dumps(
        question_for_evaluation,
        indent=2
    )
)

{
  "question": "You are running an A/B test to compare the conversion rates of two website designs. Design A received 2,000 visits with 120 conversions, and Design B received 2,500 visits with 165 conversions. Describe how you would test whether the difference in conversion rates is statistically significant, including the choice of test, assumptions, calculation steps, and interpretation of the results.",
  "topic": "Statistics",
  "difficulty": "medium",
  "question_type": "technical",
  "expected_concepts": [
    "Formulating null and alternative hypotheses for proportions",
    "Choosing an appropriate test (e.g., two\u2011proportion z\u2011test or chi\u2011square test)",
    "Checking assumptions such as independent samples and sufficient sample size for normal approximation",
    "Calculating pooled proportion, standard error, and test statistic",
    "Computing p\u2011value and comparing it to a significance level (e.g., \u03b1 = 0.05)",
    "Interpreting the result in terms of

In [28]:
evaluation_schema = AnswerEvaluation.model_json_schema()

print(json.dumps(
    evaluation_schema,
    indent=2
))

{
  "properties": {
    "overall_score": {
      "title": "Overall Score",
      "type": "number"
    },
    "technical_accuracy": {
      "title": "Technical Accuracy",
      "type": "number"
    },
    "depth": {
      "title": "Depth",
      "type": "number"
    },
    "reasoning": {
      "title": "Reasoning",
      "type": "number"
    },
    "clarity": {
      "title": "Clarity",
      "type": "number"
    },
    "communication": {
      "title": "Communication",
      "type": "number"
    },
    "confidence": {
      "title": "Confidence",
      "type": "number"
    },
    "strengths": {
      "items": {
        "type": "string"
      },
      "title": "Strengths",
      "type": "array"
    },
    "weaknesses": {
      "items": {
        "type": "string"
      },
      "title": "Weaknesses",
      "type": "array"
    },
    "should_challenge": {
      "default": false,
      "title": "Should Challenge",
      "type": "boolean"
    },
    "suggested_follow_up": {
      "default":

In [29]:
EVALUATOR_PROMPT = """
You are the Answer Evaluation Agent for InterviewHive.

You are an expert technical interviewer evaluating a candidate's
answer to an interview question.

Evaluate ONLY the candidate's answer against:
- the interview question
- the expected concepts
- the target role when relevant

Do not evaluate the candidate as a person.

EVALUATION DIMENSIONS

Score every dimension from 0 to 10.

1. technical_accuracy
   - Are the technical statements correct?
   - Penalize incorrect technical claims strongly.

2. depth
   - How thoroughly does the answer explain the concept?
   - A short but correct answer can still score well.

3. reasoning
   - Does the candidate explain why, how, trade-offs,
     implications, or decision-making?

4. clarity
   - Is the answer understandable and logically organized?

5. communication
   - Does the candidate communicate the answer effectively,
     directly, and professionally?

6. confidence
   - Evaluate how appropriately and decisively the answer
     is communicated.
   - Do not invent confidence evidence.

OVERALL SCORE

overall_score must represent the overall quality of the answer.

Technical accuracy and relevance to the question are especially
important.

SCORING GUIDE

0-2 = very poor
3-4 = weak
5-6 = acceptable
7-8 = strong
9-10 = excellent

STRENGTHS

Only mention things the candidate actually did well.

WEAKNESSES

Mention specific problems in the answer.

MISSING CONCEPTS

Only include concepts from expected_concepts that are genuinely
missing or insufficiently addressed.

Do not invent missing concepts.

CHALLENGE

Set should_challenge to true if:
- the candidate makes an incorrect technical claim,
- the answer is too vague to establish understanding,
- an important concept is misunderstood,
- or a follow-up would meaningfully test the candidate.

Otherwise set it to false.

FOLLOW-UP

If should_challenge is true, provide a useful follow-up question
targeting the biggest weakness.

If no follow-up is needed, return an empty string.

IMPORTANT OUTPUT RULES

Return ONLY valid JSON.

Use EXACTLY these field names:

{
    "overall_score": 0,
    "technical_accuracy": 0,
    "depth": 0,
    "reasoning": 0,
    "clarity": 0,
    "communication": 0,
    "confidence": 0,
    "strengths": [],
    "weaknesses": [],
    "should_challenge": false,
    "suggested_follow_up": "",
    "missing_concepts": []
}

Do NOT use alternative field names such as:
- score
- feedback
- accuracy
- follow_up
- explanation

Every field must be present.

Return ONLY the JSON object.
"""

In [30]:
def evaluate_answer(
    question,
    candidate_answer
):

    evaluation_input = {
        "target_role": blueprint["target_role"],

        "question": question["question"],

        "topic": question.get("topic", ""),

        "category": question.get("category", ""),

        "difficulty": question.get("difficulty", ""),

        "question_type": question.get("question_type", ""),

        "expected_concepts": question.get(
            "expected_concepts",
            []
        ),

        "candidate_answer": candidate_answer
    }

    prompt = f"""
Evaluate the candidate's interview answer using the provided
question and expected concepts.

QUESTION CONTEXT
{json.dumps(
    evaluation_input,
    indent=2,
    ensure_ascii=False
)}

REQUIRED JSON SCHEMA
{json.dumps(
    evaluation_schema,
    indent=2
)}

Return ONLY valid JSON matching the schema.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": EVALUATOR_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return AnswerEvaluation.model_validate(result)

In [31]:
evaluation = evaluate_answer(
    question_for_evaluation,
    candidate_answer
)

print(evaluation.model_dump())

{'overall_score': 1.0, 'technical_accuracy': 0.0, 'depth': 0.0, 'reasoning': 0.0, 'clarity': 2.0, 'communication': 2.0, 'confidence': 2.0, 'strengths': ['The answer is written clearly and uses concise language.'], 'weaknesses': ['The response does not address the A/B test statistical significance question at all.', 'It discusses XGBoost, which is unrelated to hypothesis testing for conversion rates.', 'No mention of null/alternative hypotheses, test choice, assumptions, calculations, or interpretation.'], 'should_challenge': True, 'suggested_follow_up': 'Can you describe the appropriate statistical test to compare the conversion rates of Design A and Design B, including the hypotheses, assumptions, calculation steps, and how you would interpret the result?', 'missing_concepts': ['Formulating null and alternative hypotheses for proportions', 'Choosing an appropriate test (e.g., two‑proportion z‑test or chi‑square test)', 'Checking assumptions such as independent samples and sufficient s

In [32]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [33]:
# WEIGHTS = {
#     "technical_accuracy": 0.30,
#     "depth": 0.20,
#     "reasoning": 0.20,
#     "clarity": 0.10,
#     "communication": 0.10,
#     "confidence": 0.10
# }
# def calculate_weighted_score(evaluation):

#     score = (
#         evaluation.technical_accuracy
#         * WEIGHTS["technical_accuracy"]

#         + evaluation.depth
#         * WEIGHTS["depth"]

#         + evaluation.reasoning
#         * WEIGHTS["reasoning"]

#         + evaluation.clarity
#         * WEIGHTS["clarity"]

#         + evaluation.communication
#         * WEIGHTS["communication"]

#         + evaluation.confidence
#         * WEIGHTS["confidence"]
#     )

#     return round(score, 2)


In [34]:
def create_evaluation_signal(evaluation):

    answer_state = determine_answer_state(
        evaluation.model_dump()
    )

    return {
        "overall_score": evaluation.overall_score,
        "technical_accuracy": evaluation.technical_accuracy,
        "depth": evaluation.depth,
        "reasoning": evaluation.reasoning,
        "clarity": evaluation.clarity,
        "communication": evaluation.communication,
        "confidence": evaluation.confidence,
        "state": answer_state,
        "should_challenge": evaluation.should_challenge,
        "suggested_follow_up": evaluation.suggested_follow_up,
        "missing_concepts": evaluation.missing_concepts
    }

In [35]:
signal = create_evaluation_signal(
    evaluation
)

print(
    json.dumps(
        signal,
        indent=2,
        ensure_ascii=False
    )
)

{
  "overall_score": 1.0,
  "technical_accuracy": 0.0,
  "depth": 0.0,
  "reasoning": 0.0,
  "clarity": 2.0,
  "communication": 2.0,
  "confidence": 2.0,
  "state": "technical_gap",
  "should_challenge": true,
  "suggested_follow_up": "Can you describe the appropriate statistical test to compare the conversion rates of Design A and Design B, including the hypotheses, assumptions, calculation steps, and how you would interpret the result?",
  "missing_concepts": [
    "Formulating null and alternative hypotheses for proportions",
    "Choosing an appropriate test (e.g., two‑proportion z‑test or chi‑square test)",
    "Checking assumptions such as independent samples and sufficient sample size for normal approximation",
    "Calculating pooled proportion, standard error, and test statistic",
    "Computing p‑value and comparing it to a significance level (e.g., α = 0.05)",
    "Interpreting the result in terms of statistical significance and practical significance",
    "Discussing Type 

In [36]:
state.answers.append(
    candidate_answer
)

state.evaluations.append(
    signal
)

In [37]:
print("Answers:", len(state.answers))
print("Evaluations:", len(state.evaluations))

Answers: 1
Evaluations: 1


In [38]:
next_decision = manager_decision(
    state,
    evaluation=signal,
    blueprint=blueprint
)

print(
    json.dumps(
        next_decision,
        indent=2
    )
)

{
  "action": "follow_up",
  "topic": null,
  "difficulty": "medium",
  "reason": "technical_gap"
}


In [39]:
next_question = generate_question(
    topic=next_decision["topic"],
    difficulty=next_decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)
print(next_question.question)

Given an unsorted array of integers, describe an algorithm that finds the length of the longest increasing subsequence (LIS) in O(n log n) time. Explain the main steps and why the time complexity meets the requirement.
